1. 경로 읽기

In [12]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "Tue" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "W5" / "Tue"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / 'sandbox' / 'W5' / 'Tue' / 'samples'

프로젝트 루트  : d:\hanwha-agent


2-1. PDF 파일만 찾기

In [13]:
if not SAMPLES.is_dir():
    print('재료 폴더가 없습니다. sandbox/W5/Tue/samples 에 PDF 파일을 넣으세요.')
else:
    pdfs = sorted(SAMPLES.glob('*.pdf'))
    print(f'PDF 파일 : {len(pdfs)}건')
    for p in pdfs:
        print(f'제목: {p.name} \n용량: {p.stat().st_size:>8,} 바이트')

PDF 파일 : 1건
제목: 1인 창조기업 육성에 관한 법률(법률)(제22030호)(20270330).pdf 
용량:  119,622 바이트


2-2. 이 파일이 PDF 파일이 맞는지 검토

In [14]:
PDF = next(SAMPLES.glob('*.pdf'))
PDF_MAGIC = b'%PDF'
head = PDF.read_bytes()[:4]
print(f'앞 4바이트 {head!r}  →  진짜 PDF 인가? {head == PDF_MAGIC}')

앞 4바이트 b'%PDF'  →  진짜 PDF 인가? True


2-3. 전체 파일, 읽어온 파일, 건너뛴 파일 - 검토

In [15]:
from app.rag.local_parsers import parse_pdf

NAIVE = {'.pdf': parse_pdf}

read_ok = 0
skipped = []
for p in sorted(SAMPLES.iterdir()):
    if not p.is_file():
        continue
    handler = NAIVE.get(p.suffix.lower())
    if handler is None:
        skipped.append(p.suffix.lower() or '(확장자 확인 불가)')
        continue
    read_ok += 1
print(f'훑은 파일   : {read_ok + len(skipped)}건')
print(f'읽힘       : {read_ok}건')
print(f'조용히 빠짐 : {len(skipped)}건\nㄴ>빠진 이유: 확장자 {skipped}로 확인됨')

훑은 파일   : 3건
읽힘       : 1건
조용히 빠짐 : 2건
ㄴ>빠진 이유: 확장자 ['.docx', '.hwpx']로 확인됨


3-1. 읽어온 PDF 파일 파싱

In [16]:
from app.rag.local_parsers import parse_pdf
import re

pdf = parse_pdf(PDF)

ARTICLE_COUNT = re.compile(r'^제\d+조(?:의\d+)?\(', re.MULTILINE)   # 줄 맨 앞의 "제N조(" / "제N조의M("

full_text = '\n'.join(b.text for b in pdf.blocks)
articles = ARTICLE_COUNT.findall(full_text)

print(f'블록 수    : {len(pdf.blocks)}')
print(f'쪽 수      : {pdf.page_count}')
print(f'표 수      : {pdf.table_count}')
print(f'조문 수    : {len(articles)}')
first = pdf.blocks[0]
print(f'첫 블록 kind : {first.kind}')
print(f'첫 블록 text : {first.text[:300]}')

블록 수    : 5
쪽 수      : 5
표 수      : 0
조문 수    : 24
첫 블록 kind : 조항
첫 블록 text : 법제처                                                            1                                                       국가법령정보센터
1인 창조기업 육성에 관한 법률
※ 본 법령은 공포 후 시행전 개정된 사항을 모두 포함한 법령이며, 개별 시행일별로 공포된 내용은
국가법령정보센터의 '연혁' 또는 조문 앞의 '연' 아이콘을 통해 확인할 수 있습니다.
 
1인 창조기업 육성에 관한 법률 ( 약칭: 1인창조기업법 )
[시행 2027. 3. 30


3-2. 조문 단위 파싱

In [17]:
import re
from collections import Counter
from pypdf import PdfReader
from app.integrations.ports import ParsedBlock, ParsedDoc

ARTICLE_HEAD = re.compile(r'^(제\d+조(?:의\d+)?\([^)]*\)|부칙)')   # 제1조(목적), 제15조의2(…), 부칙

def _norm(line: str) -> str:
    return re.sub(r'[\d\s]', '', line)        # 쪽 번호·공백을 빼고 비교

def parse_pdf_by_article(path) -> ParsedDoc:
    reader = PdfReader(str(path))
    pages = [(page.extract_text() or '').splitlines() for page in reader.pages]

    # 1) 머리글/꼬리글: 글자 있는 모든 쪽의 위·아래 2줄에 똑같이 나오는 줄 (빈 쪽·스캔 쪽은 비교에서 제외)
    text_pages = [lines for lines in pages if any(l.strip() for l in lines)]
    edge = Counter()
    for lines in text_pages:
        edge.update({_norm(l) for l in lines[:2] + lines[-2:] if l.strip()})
    repeated = {k for k, n in edge.items() if len(text_pages) > 1 and n == len(text_pages)}

    # 2) 조문 제목 줄에서 끊어 블록 만들기 (쪽이 넘어가도 이어 붙임)
    blocks, cur, loc = [], [], None
    def flush():
        text = '\n'.join(cur).strip()
        if text:
            blocks.append(ParsedBlock('조항' if loc else '머리말', loc or 'p.1', text))

    for page_no, lines in enumerate(pages, 1):
        for line in lines:
            line = line.strip()
            if not line or _norm(line) in repeated:
                continue
            m = ARTICLE_HEAD.match(line)
            if m:
                flush()
                cur, loc = [], f'{m.group(1)} · p.{page_no}'
            cur.append(line)
    flush()
    return ParsedDoc(blocks=blocks, page_count=len(reader.pages), table_count=0)


doc = parse_pdf_by_article(PDF)
print(f'블록 {len(doc.blocks)} · 쪽 {doc.page_count}')
for b in doc.blocks:
    print(f'{b.kind:<4} {b.locator:<40} {len(b.text):>5,}자')


블록 26 · 쪽 5
머리말  p.1                                        217자
조항   제1조(목적) · p.1                              101자
조항   제2조(정의) · p.1                              381자
조항   제3조(1인 창조기업 인정의 특례) · p.1                  200자
조항   제4조(다른 법률과의 관계) · p.1                       70자
조항   제5조(1인 창조기업 육성계획의 수립 등) · p.1              539자
조항   제6조(실태조사) · p.1                            321자
조항   제7조(종합관리시스템 구축ㆍ운영) · p.2                   146자
조항   제8조(1인 창조기업 지원센터의 지정 등) · p.2              903자
조항   제9조(지식서비스 거래지원) · p.2                      207자
조항   제10조(교육훈련 지원) · p.2                        457자
조항   제11조(기술개발 지원) · p.2                        281자
조항   제12조(아이디어의 사업화 지원) · p.3                   568자
조항   제13조(해외진출 지원) · p.3                         88자
조항   제14조(홍보사업 등) · p.3                         195자
조항   제15조(금융 지원) · p.3                          307자
조항   제15조의2(청년 및 인구감소지역 1인 창조기업의 우대) · p.3      159자
조항   제16조(전담기관 지정 등) · p.3                      543자
조항   제17조(조세에 대한 특례) · p.4        

3-3. 파싱 결과 확인

In [18]:
article_blocks = [b for b in doc.blocks if b.locator.startswith('제')]
assert len(article_blocks) == len(articles), '조문 수 불일치'
assert not any(line.startswith('법제처') for b in doc.blocks for line in b.text.splitlines()), '머리글이 남아 있음'   # 머리글 줄은 '법제처'로 시작
print(doc.blocks[11].text)   # 제11조: 2쪽→3쪽으로 넘어가는 조문이 한 블록으로 이어졌는지

제11조(기술개발 지원) ① 중소벤처기업부장관은 우수한 아이디어와 기술을 보유한 1인 창조기업을 위하여 다음 각 호
의 지원을 할 수 있다. <개정 2013. 3. 22., 2017. 7. 26.>
1. 1인 창조기업의 단독 또는 공동 기술개발
2. 1인 창조기업과 「중소기업기본법」에 따른 중소기업 간의 공동 기술개발
3. 그 밖에 1인 창조기업의 기술개발을 촉진하기 위하여 필요한 사항
② 제1항에 따른 지원의 절차와 범위 등에 필요한 사항은 대통령령으로 정한다.
[제목개정 2013. 3. 22.]


**3-4. 제출할 결과물 (ParsedDoc · 문단 개수 · 표 개수 · 문단 첫 50글자)**

In [ ]:
def report(name: str, parsed: ParsedDoc) -> None:                             # 문서 한 건의 결과 출력하는 함수
    assert isinstance(parsed, ParsedDoc), f'ParsedDoc 이 아님: {type(parsed)}' # 결과가 ParsedDoc 형태인지 확인
    paragraphs = [b for b in parsed.blocks if b.kind != '표']                 # 표가 아닌 블록-> 문단

    print(f'문서       : {name}')
    print(f'결과 형태  : {type(parsed).__name__}')
    print(f'문단 개수  : {len(paragraphs)}')
    print(f'표 개수    : {parsed.table_count}')
    print('-'*30, '문단(조항)별 첫 50글자', '-'*30)
    for i, b in enumerate(paragraphs, 1):                                     # 문단을 1번부터 하나씩
        head = ' '.join(b.text.split())[:50]                                  # 줄바꿈·연속 공백을 한 칸으로 합치고 앞 50글자
        print(f'{i:>3}. [{b.locator}] {head}')                                # 번호, 위치, 첫 50글자 출력

report(PDF.name, doc)                                                         # 3-2 에서 만든 ParsedDoc 으로 실행

문서       : 1인 창조기업 육성에 관한 법률(법률)(제22030호)(20270330).pdf
결과 형태  : ParsedDoc
문단 개수  : 26
표 개수    : 0
------------------------------ 문단(조항)별 첫 50글자 ------------------------------
  1. [p.1] ※ 본 법령은 공포 후 시행전 개정된 사항을 모두 포함한 법령이며, 개별 시행일별로 공포된
  2. [제1조(목적) · p.1] 제1조(목적) 이 법은 창의성과 전문성을 갖춘 국민의 1인 창조기업 설립을 촉진하고 그 성
  3. [제2조(정의) · p.1] 제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2013. 3. 22.
  4. [제3조(1인 창조기업 인정의 특례) · p.1] 제3조(1인 창조기업 인정의 특례) 1인 창조기업이 규모 확대의 이유로 1인 창조기업에 해
  5. [제4조(다른 법률과의 관계) · p.1] 제4조(다른 법률과의 관계) 1인 창조기업의 육성에 관하여 다른 법률에 특별한 규정이 있는
  6. [제5조(1인 창조기업 육성계획의 수립 등) · p.1] 제5조(1인 창조기업 육성계획의 수립 등) ① 중소벤처기업부장관은 1인 창조기업을 육성하기
  7. [제6조(실태조사) · p.1] 제6조(실태조사) ① 중소벤처기업부장관은 1인 창조기업을 체계적으로 육성하고 육성계획을 효
  8. [제7조(종합관리시스템 구축ㆍ운영) · p.2] 제7조(종합관리시스템 구축ㆍ운영) 중소벤처기업부장관은 1인 창조기업 관련 정보를 종합적으로
  9. [제8조(1인 창조기업 지원센터의 지정 등) · p.2] 제8조(1인 창조기업 지원센터의 지정 등) ① 정부는 1인 창조기업 및 1인 창조기업을 하
 10. [제9조(지식서비스 거래지원) · p.2] 제9조(지식서비스 거래지원) ① 중소벤처기업부장관은 1인 창조기업의 지식 서비스 거래를 활
 11. [제10조(교육훈련 지원) · p.2] 제10조(교육훈련 지원) 

4. 표 없음 표

| 표 | 열 |
|---|---|
| 행 | 셀 |